# Initialisation
Code below is for preppring tools for use

---

In [475]:
# import plotting tools
# plotnine is the python clone of ggplot2 from R
# this should be used over plt where possible

import pandas as pd
import matplotlib as plt
from plotnine import *
import numpy as np

In [476]:
from collections import *
from skimage import *
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [477]:
from transformers import AutoTokenizer, pipeline, AutoModel
import torch
import re
from thefuzz import process

# Testing
The code below is for the testing of basic premise and ideas, such as probing raw statistics and the like. This may also be used for getting familuar with certain tools and datasets.

In [478]:
df = pd.read_csv('att_sch_stats/att_sch.csv')

attainment_2025 = df[df['DateCode'].astype(str).str.contains("2024-25", na=False)]
above_5_2025 = attainment_2025[
    (
        attainment_2025['Number Of Awards'].astype(str).str.contains("5", na=False)
    ) & (
        attainment_2025['SCQF Level'].astype(str).str.contains("6", na=False)
    )
]
att_list = above_5_2025[["GeographyName", "Value"]]
att_list = att_list.sort_values(by="GeographyName")
att_list = att_list[att_list.GeographyName != "National"]

supressors = ["*", "#"]

supressed_schools = att_list[
    att_list["Value"].astype(str).isin(supressors)
].sort_values(by="GeographyName")

att_list = att_list[
    ~att_list["Value"].astype(str).isin(supressors)
].copy()

att_list

,GeographyName,Value
155091,Aberdeen Grammar School,72
155511,Aboyne Academy,46
171891,Airdrie Academy,35
154111,Alford Academy,41
169511,All Saints Secondary School,38
...,...,...
171261,Whitehill Secondary School,22
151241,Wick High School,18
174131,Williamwood High School,82
174061,Woodfarm High School,74


In [479]:
sch_sum_2025 = pd.read_excel('sch_sum_stats/sch_sum_2025.xlsx', sheet_name=2, header=1)
hs_sum_2025 = sch_sum_2025[sch_sum_2025["School Type"].astype(str).str.contains("Secondary")]
hs_sum_2025 = hs_sum_2025.drop(columns=primary_ys).drop(columns="Special school pupils")
hs_sum_2025 = hs_sum_2025.sort_values(by="School Name")
hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(["Braeview Academy", "Craigie High School", "Glasgow Gaelic School", "St Joseph's Academy", "The Community School of Auchterarder"])]

supressed_names = supressed_schools["GeographyName"].to_list()
pattern = '|'.join(re.escape(name) for name in supressed_names)
hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).str.contains(pattern, case=False)].copy()

# hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(supressed_schools["GeographyName"])].copy()
# hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"])]
att_list = att_list[att_list["GeographyName"].astype(str).isin(hs_sum_2025["School Name"].astype(str))].copy()
hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"].astype(str))].copy()

In [480]:
target = "Value"
primary_ys = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]

simd_columns = [
    "SIMD Quintile 1 [Note 6]",
    "SIMD Quintile 2 [Note 6]",
    "SIMD Quintile 3 [Note 6]",
    "SIMD Quintile 4 [Note 6]",
    "SIMD Quintile 5 [Note 6]"
]

numeric_columns = simd_columns + ["Pupil roll"]

for column in numeric_columns:
    hs_sum_2025[column] = pd.to_numeric(
        hs_sum_2025[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

for column in simd_columns:
    hs_sum_2025[f"{column} proportion"] = (
        hs_sum_2025[column] / hs_sum_2025["Pupil roll"]
    )

simd_features = [
    f"{column} proportion"
    for column in simd_columns
]


In [481]:
URC = ["Urban [Note 7]", "Small town [Note 7]", "Rural [Note 7]"]

def find_midpoint(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip().replace("%","")
    lower, upper = val.split("-")
    return (float(lower) + float(upper)) / 200

for column in URC:
    hs_sum_2025[f"{column} proportion"] = (
        hs_sum_2025[column].apply(find_midpoint)
    )

URF = [
    f"{column} proportion"
    for column in URC
]

In [482]:
denomination = "Denomination of school"

hs_sum_2025["Denomination Encoded"] = (
    hs_sum_2025[denomination]
    .map({
        "Non-denominational school" : 0,
        "Roman Catholic School" : 1
    })
)

In [483]:
features = [
    "Average class size",
    "Pupil roll",
    "Pupil Teacher Ratio",
    "Pupils with an Additional Support Need recorded",
    "Denomination Encoded",
    *simd_features,
    *URF,
]

In [484]:
hs_sum_2025

,Local Authority,SeedCode,School Name,School Type,Teachers (Full Time Equivalent) [Note 1],Pupil roll,Pupil Teacher Ratio,Average class size,Female,Male,...,Denomination of school,SIMD Quintile 1 [Note 6] proportion,SIMD Quintile 2 [Note 6] proportion,SIMD Quintile 3 [Note 6] proportion,SIMD Quintile 4 [Note 6] proportion,SIMD Quintile 5 [Note 6] proportion,Urban [Note 7] proportion,Small town [Note 7] proportion,Rural [Note 7] proportion,Denomination Encoded
6,Aberdeen City,5244439,Aberdeen Grammar School,Secondary,113.6,1369,12.1,0,684,685,...,Non-denominational school,0.027027,0.097151,0.200146,0.154127,0.521549,0.95,0.05,0.05,0.0
19,Aberdeenshire,5257530,Aboyne Academy,Secondary,55.8,631,11.3,0,301,330,...,Non-denominational school,0.000000,0.000000,0.055468,0.557845,0.386688,NaN,0.05,0.95,0.0
27,North Lanarkshire,8505438,Airdrie Academy,Secondary,77.5,1018,13.1,0,508,510,...,Non-denominational school,0.436149,0.186640,0.214145,0.144401,0.018664,0.85,0.05,0.15,0.0
39,Aberdeenshire,5230934,Alford Academy,Secondary,58.9,689,11.7,0,337,352,...,Non-denominational school,0.000000,0.000000,0.095791,0.447025,0.457184,0.05,0.05,0.95,0.0
43,Glasgow City,8431930,All Saints Secondary School,Secondary,70,955,13.6,0,487,468,...,Roman Catholic school,0.724607,0.176963,0.036649,0.041885,0.016754,0.95,0.05,0.05,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2391,Glasgow City,8439532,Whitehill Secondary School,Secondary,62.2,651,10.5,0,309,342,...,Non-denominational school,0.668203,0.138249,0.152074,0.030722,0.010753,0.95,0.05,NaN,0.0
2400,Highland,5102138,Wick High School,Secondary,58.5,719,12.3,0,374,345,...,Non-denominational school,0.207232,0.246175,0.347705,0.198887,0.000000,NaN,0.55,0.45,0.0
2404,East Renfrewshire,8602034,Williamwood High School,Secondary,137.9,1664,12.1,0,847,817,...,Non-denominational school,0.006010,0.049279,0.016827,0.129207,0.798678,0.95,0.05,0.05,0.0
2417,East Renfrewshire,8601836,Woodfarm High School,Secondary,72.5,686,9.5,0,336,350,...,Non-denominational school,0.065598,0.103499,0.112245,0.214286,0.504373,0.95,0.05,0.05,0.0


In [485]:
att_list

,GeographyName,Value
155091,Aberdeen Grammar School,72
155511,Aboyne Academy,46
171891,Airdrie Academy,35
154111,Alford Academy,41
169511,All Saints Secondary School,38
...,...,...
171261,Whitehill Secondary School,22
151241,Wick High School,18
174131,Williamwood High School,82
174061,Woodfarm High School,74


In [486]:
data = hs_sum_2025.merge(
    att_list[["GeographyName", "Value"]],
    left_on="School Name",
    right_on="GeographyName",
    how="inner",
    validate="one_to_one"
)

x = data[features]
y = pd.to_numeric(data["Value"], errors="coerce")

valid = y.notna()
x = x.loc[valid].copy()
y = y.loc[valid].copy()
numeric_features = features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features)
])

In [487]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

In [488]:
model = Pipeline([
    ("Preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 9.37430303030303
RMSE: 11.778034249732794
R²: 0.47087918871696366


In [489]:
# import matplotlib.pyplot as plt

# # 1. Calculate average Pupil Teacher Ratio per council
# avg_ptr = db.groupby("Local Authority")["Pupil Teacher Ratio"].mean().reset_index()

# # 2. Create figure and axes object properly
# fig, ax = plt.subplots(figsize=(10, 6))

# # 3. Plot a horizontal bar chart
# ax.barh(avg_ptr["Local Authority"], avg_ptr["Pupil Teacher Ratio"], color="skyblue")

# ax.set_xlabel("Average Pupil Teacher Ratio")
# ax.set_ylabel("Local Authority")
# ax.set_title("Average Pupil Teacher Ratio by Council")

# plt.tight_layout()
# plt.show()
print(hs_sum_2025[numeric_columns].isna().sum())

SIMD Quintile 1 [Note 6]    31
SIMD Quintile 2 [Note 6]    22
SIMD Quintile 3 [Note 6]     7
SIMD Quintile 4 [Note 6]    19
SIMD Quintile 5 [Note 6]    34
Pupil roll                   0
dtype: int64


In [490]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)
model_predictions = model.predict(X_test)

print("Baseline R²:", r2_score(y_test, baseline_predictions))
print("Model R²:", r2_score(y_test, model_predictions))

print("Baseline MAE:", mean_absolute_error(y_test, baseline_predictions))
print("Model MAE:", mean_absolute_error(y_test, model_predictions))

Baseline R²: -0.002397924935867657
Model R²: 0.47087918871696366
Baseline MAE: 12.91107226107226
Model MAE: 9.37430303030303
